# 第5日 ノートブック01：異常検知の考え方

# Day 5 Notebook 01: Introduction to Anomaly Detection

---

## 学習目標 / Learning objectives

| 日本語 | English |
| --- | --- |
| 1. 異常検知が何をするのか説明できる | 1. Explain what anomaly detection does |
| 2. 正常サンプルと異常サンプルを区別できる | 2. Distinguish normal from abnormal samples |
| 3. 教師あり学習との違いを説明できる | 3. Explain how this differs from supervised learning |
| 4. パッチ・特徴・異常スコアの関係を説明できる | 4. Explain patches, features and anomaly scores |
| 5. PatchCore の考え方を、比喩を使って説明できる | 5. Explain PatchCore using an analogy |

## このノートブックの基本情報 / Notebook facts

| 項目 / Item | 内容 / Details |
| --- | --- |
| 学習時間（目安） / Estimated duration | 100 分 / 100 minutes |
| 難易度 / Difficulty | 初級 / Beginner |
| 前提条件 / Prerequisites | 第4日を終えていること。 / You have finished Day 4. |
| 必要なライブラリ / Required libraries | numpy, matplotlib, pillow, opencv-python |
| 使用カーネル / Jupyter kernel | aivision |

### 提出物 / Expected deliverables

| 日本語 | English |
| --- | --- |
| 異常検知と物体検出の違いの説明 | Your explanation of how anomaly detection differs from detection |
| 異常スコアの読み方の記録 | Notes on how to read an anomaly score |

## 本日の学習タイムライン / Learning timeline for today

| 時刻 / Time | 内容（日本語） | Activity (English) |
| --- | --- | --- |
| 08:00–09:00 | 講師：異常検知の考え方 | Instructor: the idea of anomaly detection |
| 09:00–10:30 | Part 1〜Part 3（正常・異常・教師なし） | Part 1-Part 3 (normal, abnormal, unsupervised) |
| 10:30–10:40 | 休憩 | Short break |
| 10:40–12:00 | Part 4〜Part 6（パッチ・特徴・スコア） | Part 4-Part 6 (patches, features, scores) |
| 12:00–12:50 | 昼休み | Lunch break |
| 12:50–13:50 | 講師：振り返りと午後の説明 | Instructor: review and briefing |
| 13:50–16:50 | 演習・提出物 | Exercises and submission |

> **日本語：** 12:00–12:50 は昼休みです。08:00–09:00 と 12:50–13:50 は講師の時間、それ以外は自分のペースで進めてください。
>
> **English:** 12:00-12:50 is the lunch break. 08:00-09:00 and 12:50-13:50 are instructor sessions; the remaining time is self-paced work.

---

## Part 0　準備 / Preparation

In [ ]:
from pathlib import Path

def find_course_root(start=None):
    """研修フォルダ（AI_Vision_Training）を上方向に探す。
    Walk upwards until the course root folder is found."""
    current = Path(start).resolve() if start else Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "assets").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError(
        "研修フォルダが見つかりません。JupyterLab を AI_Vision_Training で起動してください。\n"
        "Course folder not found. Start JupyterLab inside the AI_Vision_Training folder."
    )

COURSE_ROOT = find_course_root()
ASSETS = COURSE_ROOT / "assets"
print("研修フォルダ / course root :", COURSE_ROOT)
print("素材フォルダ / assets      :", ASSETS)

In [ ]:
import importlib
import sys

print("Python version:", sys.version.split()[0])

REQUIRED = [
    ("numpy", "numpy"),
    ("matplotlib", "matplotlib"),
    ("PIL", "pillow"),
    ("cv2", "opencv-python"),
]

missing = []
for import_name, pip_name in REQUIRED:
    try:
        module = importlib.import_module(import_name)
        version = getattr(module, "__version__", "unknown")
        print(f"  [OK]      {pip_name:<20} {version}")
    except ImportError:
        print(f"  [MISSING] {pip_name:<20} -> pip install {pip_name}")
        missing.append(pip_name)

if missing:
    print("\n未導入のパッケージがあります / Some packages are missing:", ", ".join(missing))
    print("docs/troubleshooting/installation.md を確認してください / see that file for help.")
else:
    print("\nこのノートブックに必要なライブラリはすべて揃っています。")
    print("All libraries required by this notebook are available.")

In [ ]:
import platform
from pathlib import Path

from matplotlib import font_manager, rcParams

def find_japanese_font():
    """日本語を表示できるフォントを探す。見つからなければ None を返す。
    Locate a font that can render Japanese. Returns None when nothing is found."""
    system = platform.system()
    if system == "Windows":
        folders = [Path("C:/Windows/Fonts")]
    elif system == "Darwin":
        folders = [Path("/System/Library/Fonts"), Path("/Library/Fonts"), Path.home() / "Library/Fonts"]
    else:
        folders = [Path("/usr/share/fonts"), Path("/usr/local/share/fonts"), Path.home() / ".fonts"]

    preferred = [
        "NotoSansJP-Regular", "NotoSansCJK-Regular", "YuGothR", "meiryo", "msgothic",
        "BIZ-UDGothicR", "NotoSansJP", "HiraginoSans",
    ]
    found = []
    for folder in folders:
        if not folder.is_dir():
            continue
        for path in folder.rglob("*"):
            if path.suffix.lower() not in {".ttf", ".ttc", ".otf"}:
                continue
            name = path.stem
            if any(key.lower() in name.lower() for key in preferred):
                found.append(path)

    def rank(path):
        name = path.stem.lower()
        return (0 if "regular" in name or name.endswith("r") else 1, len(name))

    return sorted(found, key=rank)[0] if found else None


japanese_font = find_japanese_font()
if japanese_font is None:
    print("日本語フォントが見つかりません / no Japanese font found.")
    print("図のタイトルが □ になる場合は、英字のタイトルを使ってください。")
    print("If figure titles show boxes, use English titles instead.")
else:
    font_manager.fontManager.addfont(str(japanese_font))
    family_name = font_manager.FontProperties(fname=str(japanese_font)).get_name()
    rcParams["font.family"] = [family_name, "DejaVu Sans"]
    rcParams["axes.unicode_minus"] = False
    print("図の日本語フォント / figure font:", family_name)
    print("  ->", japanese_font)

In [ ]:
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt

def show(images, titles=None, figsize=(13, 4.5), cmap=None):
    if not isinstance(images, (list, tuple)):
        images = [images]
    if titles is None:
        titles = [""] * len(images)
    fig, axes = plt.subplots(1, len(images), figsize=figsize)
    if len(images) == 1:
        axes = [axes]
    for ax, image, title in zip(axes, images, titles):
        if getattr(image, "ndim", 3) == 3:
            ax.imshow(image)
        else:
            ax.imshow(image, cmap=cmap or "inferno", vmin=0, vmax=255)
        ax.set_title(title, fontsize=10)
        ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()
    plt.show()

---

## Part 1　異常検知とは / What anomaly detection is

#### 日本語：「いつもと違う」を見つける
**日本語：** **異常検知** は、**正常とはどういうものか** を学び、
そこから **外れた部分** を見つける方法です。

```text
学習：正常品の画像だけを見せる（欠陥のラベルは不要）
推論：新しい画像が、正常からどれくらい外れているかを数値で返す
```

**日本語：** 第 3 日・第 4 日の YOLO との違いはここです。

| | YOLO（物体検出） | PatchCore（異常検知） |
|---|---|---|
| 学習に必要なもの | 欠陥の位置と種類のラベル | **正常品の画像だけ** |
| 出力 | 四角 + クラス + confidence | **異常スコア + ヒートマップ** |
| 苦手なこと | 学習していない種類 | 何の欠陥かの特定 |

**English:** **Anomaly detection** learns **what normal looks like**, then finds the parts that
deviate from it.

```text
training: show normal images only (no defect labels needed)
inference: report how far a new image deviates from normal, as a number
```

That is the difference from YOLO on Days 3-4:

| | YOLO (object detection) | PatchCore (anomaly detection) |
|---|---|---|
| needs for training | defect position and class labels | **normal images only** |
| output | box + class + confidence | **anomaly score + heatmap** |
| weak at | defect types it never saw | naming which defect it is |
#### English: Finding 'not like usual'

#### 日本語：正常サンプルと異常サンプル
**日本語：**

- **正常サンプル（normal sample）** … 問題のない製品の画像
- **異常サンプル（abnormal sample）** … 欠陥のある製品の画像

**日本語：** 現場では、**正常品のほうが圧倒的に多く、集めやすい** です。
不良品は「たまにしか出ない」からです。異常検知はこの性質を利用します。

**English:**

- a **normal sample** is an image of a product with no problem
- an **abnormal sample** is an image of a defective product

On a real line, **good products are far more common and far easier to collect** — defects occur only
occasionally. Anomaly detection exploits exactly that.
#### English: Normal and abnormal samples

In [ ]:
normal_images = sorted((ASSETS / "anomaly_dataset" / "pvc_sheet" / "train" / "good").glob("*.png"))
defect_images = sorted((ASSETS / "anomaly_dataset" / "pvc_sheet" / "test").rglob("*.png"))
defect_images = [p for p in defect_images if p.parent.name != "good"]

print("正常画像 / normal training images:", len(normal_images))
print("異常画像 / abnormal test images  :", len(defect_images))
print()
print("異常画像の内訳 / breakdown:")
from collections import Counter
for name, count in sorted(Counter(p.parent.name for p in defect_images).items()):
    print(f"  {name:<16} {count}")

if normal_images:
    show([cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB) for p in normal_images[:3]],
         [f"normal {p.stem}" for p in normal_images[:3]])

---

## Part 2　教師ありと教師なし / Supervised versus unsupervised

#### 日本語：言葉の整理
**日本語：**

| 学習の種類 | データ | 例 |
|---|---|---|
| 教師あり（supervised） | 入力＋正解 | YOLO（第 4 日） |
| 教師なし（unsupervised） | 入力のみ | クラスタリング |
| 半教師あり・正常のみ学習 | 正常データのみ | **PatchCore** |

**日本語：** PatchCore は「正常だけ」で学ぶので、厳密には **教師なし** に近い方式です
（文献では one-class / semi-supervised と呼ばれることもあります）。

**日本語：** 現場での利点は明確です。**「欠陥に名前を付けて四角で囲む」作業が不要** になります。
この作業は、1 枚あたり数十秒〜数分かかり、しかも人によって判断がぶれます。

**English:**

| Learning type | Data | Example |
|---|---|---|
| supervised | input + correct answer | YOLO (Day 4) |
| unsupervised | input only | clustering |
| normal-only ("one-class") | normal data only | **PatchCore** |

Because PatchCore learns from normal data alone, it is best described as **one-class** (the
literature also calls it semi-supervised).

The practical benefit is clear: **nobody has to draw boxes and name defects**. That work costs tens
of seconds to minutes per image, and different people disagree about the result.
#### English: Sorting out the terms

---

## Part 3　パッチと特徴 / Patches and features

#### 日本語：パッチ＝画像の小片
**日本語：** PatchCore は、画像を **小さな区画（パッチ）** に分けて考えます。

```text
256 × 256 の画像 → 16 × 16 の格子 → 1 区画が 16 × 16 画素のパッチ
```

**日本語：** なぜ分けるか。**異常は画像全体ではなく、局所的に起きる** からです。
「シート全体は正常だが、この 16 × 16 の区画だけ変」という見方をします。

**English:** PatchCore thinks in **small tiles called patches**.

```text
a 256 x 256 image -> a 16 x 16 grid -> each patch covers 16 x 16 pixels
```

Why split? Because **anomalies are local**: "the whole sheet looks normal, but this one tile looks
different".
#### English: A patch is a small piece of the image

In [ ]:
example = cv2.cvtColor(cv2.imread(str(normal_images[0])), cv2.COLOR_BGR2RGB) if normal_images else None

if example is not None:
    height, width = example.shape[:2]
    grid = 8
    patch_h, patch_w = height // grid, width // grid
    print(f"image {width} x {height} -> patch {patch_w} x {patch_h} px, {grid * grid} patches")

    overlay = example.copy()
    for row in range(1, grid):
        overlay[row * patch_h, :] = [255, 0, 0]
    for col in range(1, grid):
        overlay[:, col * patch_w] = [255, 0, 0]

    show([example, overlay], ["original", f"{grid} x {grid} patches"], figsize=(11, 4.3))
else:
    print("正常画像がありません / no normal images")

#### 日本語：特徴（feature）とは
**日本語：** **特徴（feature）** は、パッチを **数値の並び** に変換したものです。

```text
パッチ（16 × 16 画素の画像） -> [0.14, 0.73, 0.08, ...]（数百〜数千個の数値）
```

**日本語：** 同じ見た目なら、似た数値の並びになります。**「見た目」を「数字の住所」に
置き換えている** と考えると分かりやすいです。

特徴を計算する部分（**バックボーン**）は、大量の画像で学習済みのものを借ります
（例：ImageNet で学習した ResNet）。

**English:** A **feature** is a patch converted into **a list of numbers**.

```text
patch (16 x 16 pixels) -> [0.14, 0.73, 0.08, ...]  (hundreds to thousands of numbers)
```

Similar appearance produces a similar list. Think of it as turning **appearance into a numeric
address**.

The part that computes features (the **backbone**) is taken from a model pretrained on a large
dataset (for example ResNet trained on ImageNet).
#### English: What a feature is

#### 日本語：最近傍（nearest neighbour）との類似度
**日本語：** 新しいパッチの特徴が、**正常パッチの特徴の集まり（メモリバンク）** と
どれくらい違うかを測ります。

```text
新しいパッチの特徴 → メモリバンクの中で一番近い正常パッチを探す
                  → その距離が「異常スコア」
```

- 距離が **小さい** … 正常に近い → **正常**
- 距離が **大きい** … 正常に無い見た目 → **異常の疑い**

**日本語：** 「一番近い正常パッチとの距離」を使うのが、PatchCore の中心的な考え方です。

**English:** Compare a new patch's feature against the **collection of normal patch features (the
memory bank)**:

```text
new patch feature -> find the closest normal patch in the memory bank
                  -> that distance is the anomaly score
```

- small distance: looks normal -> **normal**
- large distance: nothing like normal -> **suspicious**

"Distance to the nearest normal patch" is the core idea of PatchCore.
#### English: Similarity to the nearest neighbour

In [ ]:
# 2 次元の特徴で「最近傍までの距離」を体験する / experience nearest-neighbour distance in 2-D
rng = np.random.default_rng(seed=3)

memory_bank = rng.normal(loc=[0.0, 0.0], scale=0.35, size=(120, 2))   # 正常パッチの特徴
candidates = {
    "almost normal / ほぼ正常": np.array([0.15, -0.20]),
    "slightly odd  / やや変"  : np.array([0.75, 0.60]),
    "clearly odd   / 明らかに変": np.array([2.20, 1.60]),
}

plt.figure(figsize=(6, 6))
plt.scatter(memory_bank[:, 0], memory_bank[:, 1], s=14, alpha=0.5, label="normal patches (memory bank)")

for label, point in candidates.items():
    distances = np.linalg.norm(memory_bank - point, axis=1)
    nearest = distances.min()
    plt.scatter(*point, s=80, marker="X", label=f"{label}  d={nearest:.2f}")

plt.legend(fontsize=8, loc="upper left")
plt.title("anomaly score = distance to the nearest normal patch")
plt.xlabel("feature 1")
plt.ylabel("feature 2")
plt.grid(alpha=0.3)
plt.show()

print("最近傍までの距離 / distance to the nearest normal patch")
for label, point in candidates.items():
    nearest = float(np.linalg.norm(memory_bank - point, axis=1).min())
    print(f"  {label:<26} {nearest:.3f}")

### 実行結果の例 / Expected output

**日本語：** 次のような結果になります（数値は環境によって多少変わります）。

```text
最近傍までの距離 / distance to the nearest normal patch
  almost normal / ほぼ正常     0.1xx
  slightly odd  / やや変       0.4xx
  clearly odd   / 明らかに変   1.9xx
```

**English:** You should see something like the following (exact numbers vary slightly between machines).

```text
distance to the nearest normal patch
  almost normal    0.1xx
  slightly odd     0.4xx
  clearly odd      1.9xx
```

#### 日本語：グラフの読み方
**日本語：** 青い点が **正常パッチ**、× が **新しいパッチ** です。

- 正常パッチの **かたまりの中** にある × は、距離が小さい → **正常**
- かたまりから **遠い** × は、距離が大きい → **異常の疑い**

**日本語：** 実際の PatchCore では、この 2 次元が **数百〜数千次元** になります。
**次元が増えても、考え方は同じ** です。

**English:** Blue dots are **normal patches**; the X marks are **new patches**.

- An X **inside** the cloud has a small distance -> **normal**
- An X **far** from the cloud has a large distance -> **suspicious**

In real PatchCore those two dimensions become **hundreds or thousands**. **The idea does not change.**
#### English: How to read the plot

---

## Part 4　PatchCore の考え方 / The PatchCore idea

In [ ]:
show([Image.open(ASSETS / "diagrams" / "diagram_patchcore_concept.png")],
     ["PatchCore in four steps / PatchCore の 4 ステップ"], figsize=(11, 5.5))

#### 日本語：なぜ「記憶」なのか
**日本語：** PatchCore は、正常パッチの特徴を **そのまま保存（記憶）** します。
そして「一番近い正常パッチとの距離」を毎回計算します。

これは、**熟練検査員のやり方に近い** です。

> 「この模様は見たことがある。これは見たことがない。」

**日本語：** 熟練者は「良品とは何か」を頭に持っていて、目新しいものを見つけます。
PatchCore はそれを、数値の集まりとして実現しています。

**日本語：** 注意点として、正常パッチの数が膨大になるため、
**代表的なものだけを残す（コアセット）** 工夫が入っています。
`coreset_sampling_ratio` がその割合です。

**English:** PatchCore **stores the normal patch features** directly, then computes "distance to the
closest normal patch" every time.

That is close to how a skilled inspector works:

> "I have seen this pattern before. I have not seen that one."

An experienced operator carries a mental model of "what good looks like" and notices novelty.
PatchCore does the same with numbers.

Because the number of normal patches is huge, only **a representative subset (the coreset)** is
kept. `coreset_sampling_ratio` sets that fraction.
#### English: Why a memory bank

---

## Part 5　異常スコアとヒートマップ / Anomaly scores and heatmaps

#### 日本語：2 種類の出力
**日本語：** PatchCore は 2 つを返します。

| 出力 | 意味 | 使い方 |
|---|---|---|
| 異常スコア | 画像 1 枚の総合点（0〜1 に近い数値） | OK/NG の判定 |
| ヒートマップ | どの領域がどれくらい異常か（画素ごと） | 原因の確認、位置の記録 |

**日本語：** ヒートマップは **「異常の地図」** です。赤いほど異常の度合いが高いことを示します。

**English:** PatchCore returns two things:

| Output | Meaning | Use |
|---|---|---|
| anomaly score | one number per image | OK/NG decision |
| heatmap | per-pixel anomaly strength | investigating the cause, recording location |

A heatmap is **a map of anomaly**: redder means more anomalous.
#### English: Two kinds of output

In [ ]:
show([Image.open(ASSETS / "diagrams" / "diagram_anomaly_heatmap.png")],
     ["score map and heatmap (conceptual) / スコアマップとヒートマップ（概念図）"], figsize=(13, 4.5))

#### 日本語：しきい値で OK/NG を決める
**日本語：** 異常スコアは **数値** です。OK/NG にするには **しきい値** が必要です。

```text
score < しきい値  ->  OK
score >= しきい値 ->  NG
```

**日本語：** しきい値を **低く** すると NG が増えます（見逃しは減るが、誤報が増える）。
**高く** すると OK が増えます（誤報は減るが、見逃しが増える）。
これは第 3 日で見た confidence のしきい値と、まったく同じ構造です。

**English:** An anomaly score is a **number**; OK/NG needs a **threshold**.

```text
score <  threshold -> OK
score >= threshold -> NG
```

**Lowering** the threshold increases NG (fewer misses, more false alarms).
**Raising** it increases OK (fewer false alarms, more misses).
This is exactly the same structure as the confidence threshold on Day 3.
#### English: Turning a score into OK/NG with a threshold

#### 日本語：PatchCore でできないこと
**日本語：** 過信しないでください。次のことは保証されません。

1. **すべての欠陥を見つけること** — 「正常に似ている欠陥」は見逃します
2. **欠陥の種類を言うこと** — 「黒点」なのか「傷」なのかは分かりません
3. **正常なばらつきを異常と言わないこと** — 色ムラ、模様、照明のムラを異常と判定しがち

**日本語：** 3 番目は特に重要です。**正常品のばらつきを、どれだけ学習データに含めるか** が
実用上の鍵になります。

**English:** Do not over-trust it. None of the following is guaranteed:

1. **finding every defect** — a defect that looks almost normal is missed
2. **naming the defect type** — it cannot say "spot" versus "scratch"
3. **never flagging acceptable variation** — colour mottling, patterns and uneven lighting are
   easily called anomalies

Point 3 matters most: **how much normal variation you include in the training data** decides whether
this is usable in practice.
#### English: What PatchCore cannot do

---

## 演習 / Exercises

#### 日本語：演習 1（基本）：自分の言葉で説明する
**日本語：** 次の 3 つを、それぞれ 2 行以内で説明してください。

1. 異常検知と物体検出の違い
2. メモリバンクとは何か
3. 異常スコアとは何か

**English:** Explain each of the following in at most two lines: how anomaly detection differs from
object detection, what a memory bank is, and what an anomaly score is.
#### English: Exercise 1 (basic): explain it in your own words
Done when all three explanations use no more than two lines each.

In [ ]:
# 演習 1 / Exercise 1
answers = {
    "1. 異常検知と物体検出の違い / anomaly vs detection": "",
    "2. メモリバンク / memory bank": "",
    "3. 異常スコア / anomaly score": "",
}
for question, answer in answers.items():
    print(question)
    print("   ", answer or "(未記入 / not written)")
    print()

#### 日本語：演習 2（中級）：しきい値を動かす
**日本語：** 右の表を埋めてください（感覚で構いません）。

| しきい値 | NG の数 | 見逃し | 誤報 |
|---|---|---|---|
| 低い（0.2） | | | |
| 中くらい（0.5） | | | |
| 高い（0.8） | | | |

**English:** Fill in the table qualitatively.
#### English: Exercise 2 (intermediate): move the threshold
Done when you can state the direction of both error types as the threshold rises.

In [ ]:
# 演習 2 / Exercise 2
for level, threshold in [("低い / low", 0.2), ("中 / medium", 0.5), ("高い / high", 0.8)]:
    print(f"しきい値 {threshold}: NG の数=___, 見逃し=___, 誤報=___")

#### 日本語：演習 3（発展）：PVC 検査にどう使うか
**日本語：** 幅 1,300 mm・0.3 mm 欠陥・高速という条件で、
異常検知を **どの工程に、どの目的で** 入れるかを提案してください。

- YOLO と異常検知を **両方** 使うなら、順番は？
- 異常検知だけでは足りないのはなぜ？
- 最初に集めるべきデータは？

**English:** For a 1,300 mm sheet with 0.3 mm defects at high speed, propose **where and for what
purpose** you would deploy anomaly detection: in what order would you apply YOLO and anomaly
detection, why is anomaly detection alone insufficient, and what data would you collect first?
#### English: Exercise 3 (challenge): how would you use this for PVC
Done when the proposal names a purpose, an order and a first data-collection step.

In [ ]:
# 演習 3 / Exercise 3
proposal = {
    "purpose": "",
    "order_of_methods": "",
    "why_not_alone": "",
    "first_data": "",
}
for key, value in proposal.items():
    print(f"{key:<20} {value or '(未記入 / not written)'}")

---

#### 日本語：理解度チェック / Understanding questions
**日本語：**

1. 異常検知の学習に、欠陥のラベルは必要ですか。理由も書いてください。
2. なぜ画像をパッチに分けるのですか。
3. 「メモリバンク」は何を保存していますか。
4. 異常スコアが高い＝必ず不良、と言えないのはなぜですか。
5. 正常品のばらつき（色ムラなど）を学習データに含めないと、どんな問題が起きますか。

**English:**

1. Does anomaly detection need defect labels for training? Explain.
2. Why split the image into patches?
3. What does the memory bank store?
4. Why is "high anomaly score" not the same as "defective"?
5. What goes wrong if the training data excludes normal variation such as colour mottling?
#### English: Understanding questions
Model answers: `docs/instructor/day05_instructor_guide.md`.

---

## 困ったときは / Troubleshooting

**日本語：** エラーが出たら、まず「どのセルで」「どの行で」「どんなメッセージが出たか」を 書き留めてください。提出物にも記録します。

**English:** When an error appears, first write down *which cell*, *which line*, and *what the message says*. You will also record it in your submission.

| 症状（日本語） | Symptom (English) | 対処（日本語） | What to do (English) |
| --- | --- | --- | --- |
| 正常画像が表示されない | Normal images do not appear | `python assets\scripts\prepare_datasets.py --only anomaly` でデータを生成してください。 | Generate the data with `prepare_datasets.py --only anomaly`. |
| グラフの点が重なって見えない | The scatter points overlap | 正常パッチは 120 点あります。`alpha` を下げるか `s` を小さくしてください。 | There are 120 normal points; lower `alpha` or `s`. |
| 「距離」の意味が分からない | The meaning of 'distance' is unclear | 多次元の数値の差を 1 つの数値にしたものです。2 次元の例で確認してください。 | It compresses a multi-dimensional difference into one number; confirm it with the 2-D example. |
| しきい値をどう決めればよいか分からない | I do not know how to choose the threshold | 正常品だけを流してスコアを測り、その上限を目安にします。第5日02 で行います。 | Measure scores on known-good products and use their upper range. Day 5 Notebook 02 does this. |

## 完了チェックリスト / Completion checklist

**日本語：** すべてにチェックが付いたら、このノートブックは完了です。

**English:** This notebook is complete when every box is checked.

- [ ] Part 1〜Part 5 を読み、すべてのコードセルを実行した
      - [ ] Read Parts 1-5 and ran every code cell
- [ ] 正常画像と異常画像の数を確認した
      - [ ] Checked the counts of normal and abnormal images
- [ ] パッチ分割の図を確認した
      - [ ] Inspected the patch-grid figure
- [ ] 最近傍距離のグラフを確認した
      - [ ] Inspected the nearest-neighbour plot
- [ ] 演習 1〜3 を実施した
      - [ ] Completed exercises 1-3
- [ ] 理解度チェックの 5 問に答えた
      - [ ] Answered the 5 questions
- [ ] ノートブックを保存した
      - [ ] Saved the notebook

## さらに学ぶ / Further learning（任意 / optional）

### 参考リンク / Reference links

- [Anomalib ドキュメント](https://anomalib.readthedocs.io/)
- [PatchCore 論文（Roth et al., 2022）](https://arxiv.org/abs/2106.08265)
- [MVTec AD データセット](https://www.mvtec.com/company/research/datasets/mvtec-ad)

### 追加チャレンジ / Extra challenge（必須ではありません / not mandatory）

**日本語：** PatchCore の論文の図 1 を見て、メモリバンクがどこにあるかを探してください。

**English:** Look at Figure 1 of the PatchCore paper and identify where the memory bank lives.